# Peak Domain Identification with pySpectrum

pySpectrum identifies statistically significant signal regions by convolving the spectrum with a zero-area kernel (Gaussian second derivative / Mexican hat). The convolution response at each channel equals the signal-to-noise ratio (SNR) for a Gaussian peak of the local detector width. Regions where the SNR exceeds a threshold are expanded into contiguous `Domain` objects.

This approach is robust to varying peak widths because the kernel width tracks the detector resolution calibration point-by-point.

## Workflow
1. Load a time–channel list-mode spectrum
2. Set up the SNR convolution and peak finder
3. Identify all peak domains across the spectrum
4. Inspect individual detected domains

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pyspectrum.core import Spectrum
from pyspectrum.calibration import AxisCalibration, ResolutionCalibration
from pyspectrum.calibration.models.hpge_fwhm_model import StandardHPGeFWHMModel
from pyspectrum.io import TimeChannelParser
from pyspectrum.identification import Convolution
from pyspectrum.identification.snr import SNRFinder
from pyspectrum.identification.kernels.mexican_hat import gaussian_2_dev

## 1. Load spectrum from list-mode file

> **Adapt:** replace `path`, the calibration polynomial, FWHM, and `num_of_channels` with values for your detector.

## 2. Set up SNR convolution and peak finder

The `Convolution` applies the kernel locally — the kernel width at each channel is determined by `resolution(axis_value)`, so no resampling is needed even for non-linear energy calibrations.

> **Adapt `n_sigma_signal_threshold`** to trade sensitivity against false positives. Typical values: 3–5σ for high-purity spectra, 2–3σ for low-statistics measurements.

In [12]:
path = '../Library/time_channel_to_spectrum.txt'
t_c_parser = TimeChannelParser()
data = pd.read_csv(path, names =['time', 'channel', 'flag'], sep=' ', skiprows=5, usecols=range(3))

In [ ]:
# ── Adapt to your detector ────────────────────────────────────────────────────
path                   = '../Library/Doppler_broadening_Spectrum.txt'
energy_calib_poly      = np.poly1d([0.0408976444, 0.0822321508])  # channel → keV
energy_resolution_fwhm = 1.05   # FWHM at 511 keV [keV]
num_of_channels        = 16384
# ─────────────────────────────────────────────────────────────────────────────

energy_calib   = AxisCalibration(func=energy_calib_poly, name="energy")
estimated_FWHM = StandardHPGeFWHMModel().generator((0, energy_resolution_fwhm / 511**0.5, 0))
res_calib      = ResolutionCalibration(func=estimated_FWHM)

spectrum = TimeChannelParser.from_file(
    path, axis_calib=energy_calib, resolution_calib=res_calib,
    num_of_channels=num_of_channels, chunk_size=10_000,
    sep=' ', skiprows=5, names=['time', 'channel', 'flag'], usecols=[0, 1, 2],
)

In [ ]:
# ── Adapt detection thresholds ────────────────────────────────────────────────
n_sigma_signal     = 4.0
n_sigma_background = 2.0
persistence        = 0.5
# ─────────────────────────────────────────────────────────────────────────────

conv   = Convolution(resolution=res_calib.apply, kernel=gaussian_2_dev, window_fwhm=3)
finder = SNRFinder(convolution=conv,
                   n_sigma_signal_threshold=n_sigma_signal,
                   n_sigma_bg_threshold=n_sigma_background,
                   persistence_factor=persistence)

peaks_domains = finder.find(spectrum)
print(f"Detected {len(peaks_domains)} peak domains")

In [ ]:
## 3. Visualise detected domains

Detected domains are highlighted in red on the spectrum. Each domain spans the region where the SNR exceeds the background threshold for a persistence window of `persistence_factor × FWHM`.

In [ ]:
spectrum.data.plot(yscale='log', color='steelblue', label='spectrum')
for domain in peaks_domains:
    domain.data.plot(color='tomato')   # highlighted domains in red

plt.xlim([300, 650])
plt.title('Detected peak domains — Doppler broadening spectrum')
plt.xlabel('Energy [keV]')
plt.ylabel('Counts')
plt.grid(True, which='both')
plt.tight_layout()
plt.show()